# Credit Card Fraud Detection — EDA

Exploratory analysis of the Kaggle dataset before any modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

df = pd.read_csv('../data/creditcard.csv')
df.shape

## 1. Basic Info

In [ ]:
df.info()
df.describe()

In [ ]:
# Missing values
df.isnull().sum().sum()

## 2. Class Imbalance

In [ ]:
counts = df['Class'].value_counts()
print(counts)
print(f'Fraud rate: {counts[1]/len(df)*100:.3f}%')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Legit', 'Fraud'], counts.values, color=['steelblue', 'tomato'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, ax in zip([0, 1], ['steelblue', 'tomato'], axes):
    subset = df[df['Class'] == label]['Amount']
    title = 'Legit' if label == 0 else 'Fraud'
    ax.hist(subset, bins=50, color=color, edgecolor='white')
    ax.set_title(f'Amount — {title}')
    ax.set_xlabel('Amount ($)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Time Distribution

In [ ]:
df['Hour'] = (df['Time'] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, color, ax in zip([0, 1], ['steelblue', 'tomato'], axes):
    subset = df[df['Class'] == label]['Hour']
    title = 'Legit' if label == 0 else 'Fraud'
    ax.hist(subset, bins=24, color=color, edgecolor='white')
    ax.set_title(f'Hour of Day — {title}')
    ax.set_xlabel('Hour')

plt.tight_layout()
plt.show()

## 5. Feature Distributions: V1–V10

In [ ]:
v_cols = [f'V{i}' for i in range(1, 11)]
fig, axes = plt.subplots(2, 5, figsize=(18, 6))

for ax, col in zip(axes.flatten(), v_cols):
    df[df['Class'] == 0][col].plot.kde(ax=ax, label='Legit', color='steelblue')
    df[df['Class'] == 1][col].plot.kde(ax=ax, label='Fraud', color='tomato')
    ax.set_title(col)
    ax.legend(fontsize=7)

plt.suptitle('V1–V10 KDE: Legit vs Fraud', y=1.02)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap (sample)

In [ ]:
# Correlation with Class label — which V features are most predictive?
corr = df.drop(columns=['Hour']).corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(5, 8))
corr.plot.barh(ax=ax, color=['tomato' if v < 0 else 'steelblue' for v in corr.values])
ax.set_title('Correlation with Class (Fraud=1)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Key Takeaways

- **Extreme imbalance**: ~0.17% fraud — accuracy is useless as a metric; use PR-AUC.
- **Amount**: fraud transactions tend to be smaller on average — worth engineering.
- **V features**: V14, V12, V10, V4 show strongest negative correlation with fraud; V11 positive.
- **Time**: fraud has a slightly different hourly distribution — hour-of-day is a useful feature.
- **No missing values** — clean dataset, no imputation needed.